# 匯入套件

In [1]:
import os
import torch
import sys
import os
from tqdm import tqdm
import pickle
import numpy as np

sys.path.append('../')

from core.model import *
from core.train import *
from utils.checkpoint import *
from utils.util import *
from accelerate import Accelerator
from preprocessing.data_mimiciv import data_perpare

# 模型參數

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

args = parse_args()

args.modeltype = 'TS_Text'
args.fp16 = True
args.num_train_epochs = 8
args.train_batch_size = 2
args.eval_batch_size = 8

args.notes_order = 'Last'
args.num_of_notes = 5
args.output_dir = "../run/TS_Text"
args.file_path = '../../../../datasets/mimic-iv/TS_Text'
args.task = 'ihm-48-notes'

args.layers = 1
args.embed_dim = 128
args.num_modalities = 2
args.num_labels = 2
args.tt_max = 48
args.num_heads = 8
args.embed_time = 64
args.kernel_size = 1

args.irregular_learn_emb_ts = 'mTAND'
args.irregular_learn_emb_text = 'mTAND'
args.irregular_learn_emb_cxr = 'mTAND'
args.irregular_learn_emb_ecg = 'mTAND'

args.cross_method = "moe"
args.gating_function = ["laplace"]
args.num_of_experts = [16, 5]
args.top_k = [4, 4]
args.disjoint_top_k = 2
args.hidden_size = 512
args.router_type = 'joint'

args.TS_mixup = True
args.mixup_level = 'batch'
args.reg_ts = True

# 儲存預測結果、分類特徵

In [ ]:
accelerator = Accelerator(cpu=args.cpu)
device = accelerator.device
os.makedirs(args.output_dir, exist_ok=True)

_, _, test_data_loader = data_perpare(args, 'test')

make_save_dir(args)

model = MULTCrossModel(args=args,device=device,orig_d_ts=30, orig_reg_d_ts=60, orig_d_txt=768, ts_seq_num=args.tt_max, text_seq_num=args.num_of_notes)

# Hook 分類特徵
features = []
def hook(module, input, output):
    features.append(input[0].cpu())

model.proj1.register_forward_hook(hook)

model, test_data_loader = \
accelerator.prepare(model, test_data_loader)

rootdir = args.ck_file_path
seeds = range(1, 6)
all_pred_list = []
all_label_list = []

for seed in seeds:
    for subdir, dirs, files in os.walk(rootdir):
        substr = subdir.split('/')[-1]
        if 'f1' not in substr:
            continue

        file = f'{seed}.pth.tar'
        file_path = os.path.join(subdir, file)
        print(file_path)
        checkpoint = torch.load(file_path, map_location=device)
        model.load_state_dict(checkpoint['network'])
        model.eval()

        all_logits = []
        all_label = []
        
        for idx, batch in enumerate(tqdm(test_data_loader)):
            labels = batch.pop('labels')

            with torch.no_grad():
                logits = model(**batch)

            if logits.dim() == 1:
                logits = logits.unsqueeze(-1) # [B, 1]

            all_logits.append(logits.cpu().numpy())
            all_label.append(labels.cpu().numpy())
            
            pred_prob = logits[0].item()
            pred_cls = 1 if pred_prob > 0.5 else 0
            true_cls = labels[0].item()

        all_logits = np.concatenate(all_logits, axis=0)
        all_label = np.concatenate(all_label, axis=0)
        all_pred = np.where(all_logits > 0.5, 1, 0)
        all_pred_list.append(all_pred)
        all_label_list.append(all_label)

features = torch.cat(features, dim=0).numpy()
all_pred_list = np.concatenate(all_pred_list, axis=0)
all_label_list = np.concatenate(all_label_list, axis=0)

with open(rootdir + "/sample_result.pkl","wb") as f:
    result = {'features': features, 'preds': all_pred_list, 'labels': all_label_list}
    pickle.dump(result, f)

Using ../../../../datasets/mimic-iv/TS_Text/test_ihm-48-notes_stays.pkl
../run/TS_Text/ihm-48-notes_TS_Text_TS_mTAND_64_Text_mTAND_64_layer1_moe_['laplace']_joint_[16, 5]_top_[4, 4]_batch_0.0004_8_8_128_1_2_512/f1/1.pth.tar


  0%|          | 0/601 [00:00<?, ?it/s]/mnt/data/yihua/master/implementations/FuseMoE/src/analysis/../core/module.py:448: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  positions.append(torch.tensor(torch.arange(length),dtype=torch.long).to(self.device))
100%|██████████| 601/601 [00:34<00:00, 17.42it/s]


../run/TS_Text/ihm-48-notes_TS_Text_TS_mTAND_64_Text_mTAND_64_layer1_moe_['laplace']_joint_[16, 5]_top_[4, 4]_batch_0.0004_8_8_128_1_2_512/f1/2.pth.tar


100%|██████████| 601/601 [00:35<00:00, 16.77it/s]


../run/TS_Text/ihm-48-notes_TS_Text_TS_mTAND_64_Text_mTAND_64_layer1_moe_['laplace']_joint_[16, 5]_top_[4, 4]_batch_0.0004_8_8_128_1_2_512/f1/3.pth.tar


...

../run/TS_Text/ihm-48-notes_TS_Text_TS_mTAND_64_Text_mTAND_64_layer1_moe_['laplace']_joint_[16, 5]_top_[4, 4]_batch_0.0004_8_8_128_1_2_512/f1/4.pth.tar


100%|██████████| 601/601 [00:36<00:00, 16.50it/s]


../run/TS_Text/ihm-48-notes_TS_Text_TS_mTAND_64_Text_mTAND_64_layer1_moe_['laplace']_joint_[16, 5]_top_[4, 4]_batch_0.0004_8_8_128_1_2_512/f1/5.pth.tar


100%|██████████| 601/601 [00:36<00:00, 16.57it/s]


: 